# Baselines clássicos com features de intervalos R-R (Random Forest e SVM)

Mesmo split por paciente (verificado contra a referência). Picos R detectados no sinal Butterworth (estilo Pan–Tompkins); features de variabilidade R-R; τ por Youden nas predições out-of-fold do treino (10-fold agrupado por paciente).


## 1. Imports e configuracao

In [ ]:
# ── Determinismo (v3): deve ser definido ANTES do primeiro import do torch ──
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# ── Configuração do estudo OAT (lida de variáveis de ambiente) ───────────
OAT_PARAM = os.environ.get('OAT_PARAM', 'reference')   # reference|gamma|alpha|dropout|lr|wd|seed
OAT_VALUE = os.environ.get('OAT_VALUE', '')
OAT_NAME  = os.environ.get('OAT_NAME',  'reference_seed42')
OAT_ROOT  = os.environ.get('OAT_ROOT',  '/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Sensibilidade_OAT')
REF_DIR   = os.environ.get('OAT_REF_DIR', '/home/jose_alberto/Modelos_IEEE_Artigo_Revisao/Ablation_LOESS_NLM_v3_ok')

VARIANT_LABEL = f'Ablation #1 OAT - {OAT_NAME}'
VARIANT_ID    = f'ABL1_OAT_{OAT_NAME}'
print(f'OAT: param={OAT_PARAM} value={OAT_VALUE!r} name={OAT_NAME}')

import os, pickle, glob, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, roc_curve, auc as sk_auc)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SPLIT_SEED = 42                                   # partições e folds: FIXOS em todas as variações
SEED = int(os.environ.get('OAT_SEED', '42'))      # semente de treino (varia só no estudo de sementes)
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

# ── Determinismo (v3) ───────────────────────────────────────────────────
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
torch.use_deterministic_algorithms(True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def seed_worker(worker_id):
    ws = torch.initial_seed() % 2**32
    np.random.seed(ws); random.seed(ws)

set_seed(SEED)
print(f'torch {torch.__version__} | cudnn.deterministic=True | '
      f'use_deterministic_algorithms=True | CUBLAS_WORKSPACE_CONFIG={os.environ["CUBLAS_WORKSPACE_CONFIG"]}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

FS = 500; N_POINTS = 5000; N_ACF_LAGS = 5000

# ── Variante arquitetural (estudo de arquiteturas alternativas) ──────────
#   dual         : modelo de referência (CNN + ACF + attention gate)
#   cnn_only     : apenas o ramo CNN (sem ACF)
#   no_gate      : CNN + ACF com concatenação simples (sem attention gate)
#   early_fusion : sinal e ACF empilhados como 2 canais de uma única CNN
#   acf_trunc    : ACF truncada aos lags 0,4–2,0 s (200–1000 amostras)
#   mobilenet    : baseline MobileNet-1D (convoluções separáveis), só o sinal
#   lstm         : baseline recorrente compacto (CNN rasa + LSTM), só o sinal
ARCH = os.environ.get('OAT_ARCH', 'dual')
assert ARCH in ('dual', 'cnn_only', 'no_gate', 'early_fusion', 'acf_trunc', 'mobilenet', 'lstm'), ARCH
ACF_LAG_START, ACF_LAG_END = 200, 1000           # usado apenas em acf_trunc
if ARCH == 'acf_trunc':
    N_ACF_LAGS = ACF_LAG_END - ACF_LAG_START + 1  # 801
USE_ACF_INPUT = ARCH not in ('cnn_only', 'mobilenet', 'lstm')               # modelo exportado recebe a ACF?
print(f'Arquitetura: {ARCH} | N_ACF_LAGS={N_ACF_LAGS} | entrada ACF no modelo: {USE_ACF_INPUT}')

def KIN(xr, xa):
    """Lista de entradas do modelo Keras/LiteRT conforme a variante."""
    return [xr, xa] if USE_ACF_INPUT else [xr]
N_FOLDS      = 10
BATCH_SIZE   = 64
EPOCHS       = 80
LR_INITIAL   = 4e-3
PATIENCE     = 15
HOLDOUT_FRAC = 0.15   # alvo nominal; fração real ≈14,3% (StratifiedGroupKFold, 7 folds)
CALIB_FRAC   = 0.05   # 5%  calibração INT8 (split formal, separado antes do K-Fold)

# ── Splits patient-aware ──────────────────────────────────────────────
# Number of folds chosen to approximate the original 15%/5% fractions:
#   7 folds  → 1 fold ≈ 14.3% (holdout)
#   17 folds → 1 fold ≈ 5.9% do trainval ≈ 5.0% do pool (calibração INT8)
N_SPLITS_HOLDOUT = 7
N_SPLITS_CALIB   = 17

AUG_PROB = 0.30; AUG_SNR_MIN = 20; AUG_SNR_MAX = 40; AUG_AMP_RANGE = 0.15
FOCAL_GAMMA = 2.0; FOCAL_ALPHA = 0.80
DROPOUT_HEAD = 0.3; DROPOUT_CLS = 0.2; WEIGHT_DECAY = 1e-4

# ── Aplica a variação OAT (um hiperparâmetro por vez; demais na referência) ──
if   OAT_PARAM == 'gamma':   FOCAL_GAMMA  = float(OAT_VALUE)
elif OAT_PARAM == 'alpha':   FOCAL_ALPHA  = float(OAT_VALUE)
elif OAT_PARAM == 'dropout': DROPOUT_HEAD, DROPOUT_CLS = (float(v) for v in OAT_VALUE.split('/'))
elif OAT_PARAM == 'lr':      LR_INITIAL   = float(OAT_VALUE)
elif OAT_PARAM == 'wd':      WEIGHT_DECAY = float(OAT_VALUE)
elif OAT_PARAM in ('seed', 'reference', 'arch'): pass
else: raise ValueError(f'OAT_PARAM desconhecido: {OAT_PARAM}')
print(f'gamma={FOCAL_GAMMA} alpha={FOCAL_ALPHA} dropout={DROPOUT_HEAD}/{DROPOUT_CLS} '
      f'lr={LR_INITIAL} wd={WEIGHT_DECAY} train_seed={SEED} split_seed={SPLIT_SEED}')

# ── NOVO diretório de saída (não sobrescreve o original) ──────────────
BASE = f'{OAT_ROOT}/{OAT_NAME}/'
os.makedirs(BASE, exist_ok=True)
FINAL_MODEL_PATH  = f'{BASE}/ablation_loess_nlm_final.pt'
SCALER_PATH       = f'{BASE}/ablation_loess_nlm_scaler.pkl'
THRESH_PATH       = f'{BASE}/ablation_loess_nlm_threshold.pkl'
TFLITE_PATH       = f'{BASE}/ablation_loess_nlm_int8.tflite'
C_HEADER_PATH     = f'{BASE}/ablation_loess_nlm_model.h'
HOLDOUT_CSV_PATH  = f'{BASE}/ablation_loess_nlm_holdout.csv'
CALIB_IDX_PATH    = f'{BASE}/ablation_loess_nlm_calib_indices.pkl'
ROC_PATH          = f'{BASE}/ablation_loess_nlm_roc.png'
RESULTS_PATH      = f'{BASE}/ablation_loess_nlm_results.png'
WAVE_PLOT_PATH    = f'{BASE}/ablation_loess_nlm_waves.png'

# ── Paths dos datasets ─────────────────────────────────────────────────
# Os sinais serao filtrados apenas com Butterworth (HP 0.5 Hz / LP 40 Hz)
# Ablation: LOESS e NLM removidos
PATHS_CH = {
    'AFIB': '/mnt/windows/Doutorado/Datasets/Chinese_Lead_I_AFIB/',
    'SR':   '/mnt/windows/Doutorado/Datasets/Chinese_Lead_I_SR/',
}
PATHS_PTB = {
    'AFIB': '/mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_AFIB/',
    'SR':   '/mnt/windows/Doutorado/Datasets/PTB_XL_Lead_I_SR/',
}

# ── Metadados PTB-XL (mapeamento ecg_id → patient_id) ─────────────────
PTBXL_DB_CSV = '/mnt/windows/Doutorado/Datasets/PTB_XL_Raw/ptbxl_database.csv'

C_AFIB = '#D85A30'; C_SR = '#185FA5'
print(f'HOLDOUT_FRAC : {HOLDOUT_FRAC}  (alvo nominal; real ≈14,3% patient-aware)')
print(f'CALIB_FRAC   : {CALIB_FRAC}  ({int(CALIB_FRAC*100)}% calibracao INT8 formal)')
print(f'SPLITS       : StratifiedGroupKFold — holdout={N_SPLITS_HOLDOUT} folds, calib={N_SPLITS_CALIB} folds')
print(f'  CH  AFIB -> {PATHS_CH["AFIB"]}')
print(f'  CH  SR   -> {PATHS_CH["SR"]}')
print(f'  PTB AFIB -> {PATHS_PTB["AFIB"]}')
print(f'  PTB SR   -> {PATHS_PTB["SR"]}')
print(f'  PTB-XL DB -> {PTBXL_DB_CSV}')
print(f'  BASE      -> {BASE}')

## 2. Funcoes auxiliares

## 2b. Pipeline de Filtros Python - replica do firmware ESP32-S3 (ABLATION SEM LOESS+NLM)

**LOESS Smoothing e NLM Denoising REMOVIDOS** nesta ablação. Apenas Butterworth bandpass.

| Estagio | Arquivo C              | Status                                              |
|---------|------------------------|-----------------------------------------------------|
| 1       | `butterworth_filter.c` | Passa-banda zero-phase: HP 0.5 Hz + LP 40 Hz, fs=500 Hz |
| 2       | `loess_filter.c`       | **REMOVIDO** nesta ablação                          |
| 3       | `nlm_filter.c`         | **REMOVIDO** nesta ablação                          |


In [ ]:
# =============================================================================
# PIPELINE DE FILTROS - replica Python do firmware ESP32-S3
# Referencias: butterworth_filter.c, loess_filter.c, nlm_filter.c
# =============================================================================

from scipy.signal import sosfilt
import math as _math

# ── 1. Butterworth bandpass zero-phase ────────────────────────────────────────
#
# Replica butterworth_filter.c:
#   - 2a ordem Butterworth HP 0.5 Hz cascateado com LP 40 Hz
#   - Forward-backward (zero-phase) - equivalente a filtfilt de 4a ordem
#
# Coeficientes calculados pela mesma BLT com pre-warping de frequencia
# (compute_hp_coeffs / compute_lp_coeffs). sosfilt two-pass manual
# com reset de estado entre passes replica dsps_biquad_f32 forward+backward.

def _butter_sos_blt(fc, btype, fs=500.0):
    """Coeficientes SOS via BLT com pre-warping - replica compute_hp/lp_coeffs."""
    omega_a = 2.0 * fs * _math.tan(_math.pi * fc / fs)
    k  = (2.0 * fs) / omega_a
    k2 = k * k
    sqrt2k = _math.sqrt(2.0) * k
    a0 = k2 + sqrt2k + 1.0
    if btype == 'high':
        b0 =  k2 / a0
        b1 = -2.0 * k2 / a0
        b2 =  k2 / a0
    else:  # 'low'
        b0 = 1.0 / a0
        b1 = 2.0 / a0
        b2 = 1.0 / a0
    a1 =  2.0 * (1.0 - k2) / a0
    a2 = (k2 - sqrt2k + 1.0) / a0
    # Formato SOS scipy: [b0, b1, b2, 1, a1, a2]
    return np.array([[b0, b1, b2, 1.0, a1, a2]], dtype=np.float64)

_SOS_HP = _butter_sos_blt(0.5,  'high', fs=500.0)  # pre-computado
_SOS_LP = _butter_sos_blt(40.0, 'low',  fs=500.0)  # pre-computado

def butterworth_apply(signal):
    """Bandpass zero-phase: HP 0.5 Hz -> LP 40 Hz, forward-backward.

    Replica butterworth_filter_apply() com 4 chamadas a dsps_biquad_f32:
      Forward : HP(signal) -> scratch -> LP(scratch) -> signal
      Reverse : signal[::-1]
      Backward: HP -> LP (mesmo esquema, state zerado)
      Reverse : restaura ordem causal
    """
    s = signal.astype(np.float64).copy()

    def _one_pass(x):
        # state zerado a cada pass (replica memset do firmware)
        y1 = sosfilt(_SOS_HP, x)
        y2 = sosfilt(_SOS_LP, y1)
        return y2

    s = _one_pass(s)           # Forward
    s = s[::-1].copy()         # Reverse
    s = _one_pass(s)           # Backward
    s = s[::-1].copy()         # Restore order
    return s.astype(np.float32)


# ── 2. LOESS robusto ──────────────────────────────────────────────────────────
#
# Replica loess_filter.c:
#   LOESS_SPAN        = 151 amostras (half_span = 75)
#   Ajuste local de reta (1a ordem) por WLS com peso tricube
#   2 passes: 1o sem mascara, 2o com mascara MAD de outliers
#   Subtrai a tendencia estimada do sinal (baseline correction)
#   LOESS_OUTLIER_SIGMA = 6.0  (sem fator de consistencia no threshold)

LOESS_SPAN      = 151
LOESS_HALF_SPAN = (LOESS_SPAN - 1) // 2   # 75
LOESS_OUTLIER_S = 6.0

def _tricube(u):
    """w(u) = (1 - |u|^3)^3  para |u| < 1 - replica tricube() em C."""
    u = np.abs(u)
    out = np.zeros_like(u)
    m = u < 1.0
    t = 1.0 - u[m] ** 3
    out[m] = t ** 3
    return out

def _loess_fit_at(signal, i, outlier_mask=None):
    """Ajuste linear local WLS no ponto i - replica loess_fit_at().

    Parametros do firmware:
      start/end clampados nas bordas
      max_dist = max(i - start, end - i, 1)
      u = |j - i| / max_dist
      w = tricube(u)
    Retorna beta_0 (intercepto = valor ajustado em t=0).
    """
    N = len(signal)
    start = max(0, i - LOESS_HALF_SPAN)
    end   = min(N - 1, i + LOESS_HALF_SPAN)
    max_dist = max(i - start, end - i, 1)

    j = np.arange(start, end + 1)
    if outlier_mask is not None:
        j = j[outlier_mask[j] == 0]
    if len(j) == 0:
        return signal[i]

    dist = np.abs(j - i) / max_dist
    w = _tricube(dist)
    t = (j - i).astype(np.float64)
    y = signal[j].astype(np.float64)

    sw   = np.sum(w)
    swx  = np.sum(w * t)
    swy  = np.sum(w * y)
    swxx = np.sum(w * t * t)
    swxy = np.sum(w * t * y)

    denom = sw * swxx - swx * swx
    if denom < 1e-9:
        return swy / sw if sw > 0 else signal[i]
    return (swy * swxx - swx * swxy) / denom

def loess_apply(signal):
    """Dois passes LOESS robusto - replica loess_filter_apply().

    Passe 1: ajuste sem mascara -> residuos -> MAD -> mascara
    Passe 2: reajuste com mascara
    Saida: signal - trend2  (baseline correction)
    """
    N = len(signal)
    sig = signal.astype(np.float64)

    # ── Passe 1 ──────────────────────────────────────────────────────────
    trend = np.array([_loess_fit_at(sig, i) for i in range(N)])
    residuals = sig - trend

    # Threshold MAD - replica exata do firmware (sem LOESS_MAD_FACTOR)
    # loess_filter.c usa: threshold = LOESS_OUTLIER_SIGMA * median(|residuals|)
    mad = np.median(np.abs(residuals))
    threshold = LOESS_OUTLIER_S * mad
    outlier_mask = (np.abs(residuals) > threshold).astype(np.uint8)

    # ── Passe 2 ──────────────────────────────────────────────────────────
    trend2 = np.array([_loess_fit_at(sig, i, outlier_mask) for i in range(N)])

    return (sig - trend2).astype(np.float32)


# ── 3. NLM (Non-Local Means) - Darbon fast ───────────────────────────────────
#
# Replica nlm_filter.c:
#   NLM_PATCH_HALF  = 13  (p)
#   NLM_SEARCH_HALF = 694 (M)
#   NLM_K_LAMBDA    = 1.5
#   NLM_MAD_FACTOR  = 1.4826
#   sigma_hat = MAD_FACTOR * median(|diff(x)|) / sqrt(2)
#   h = 2 * NLM_PATCH_LEN * lambda^2
#   Loop: D_delta[i] = (x[i+d]-x[i])^2, prefix sum, patch distance, exp(-d2/h)
#   Auto-similaridade (delta=0): w=1 por construcao

NLM_PATCH_HALF  = 13
NLM_PATCH_LEN   = 2 * NLM_PATCH_HALF + 1   # 27
NLM_SEARCH_HALF = 694
NLM_K_LAMBDA    = 1.5
NLM_MAD_FACTOR  = 1.4826

def _nlm_sigma(signal):
    """sigma_hat via MAD das primeiras diferencas - replica estimate_noise_sigma()."""
    diffs = np.abs(np.diff(signal.astype(np.float64)))
    return NLM_MAD_FACTOR * np.median(diffs) / np.sqrt(2.0)

def nlm_apply(signal):
    """NLM Darbon fast - replica nlm_filter_apply() — loop interno vetorizado.

    Logica identica ao firmware C e a versao escalar anterior:
      D_delta[i] = (signal[i+delta] - signal[i])^2
      cum_sum    = cumsum com padding de borda
      d2(i,j)    = cum[i+p] - cum[i-p-1]   (integral sobre janela de patch)
      w          = exp(-d2 / h)
      Acumulacao simetrica sobre todos os pares (i, i+delta) simultaneamente
    O loop `for i in range(N)` foi substituido por operacoes NumPy vetorizadas,
    mantendo resultado numericamente identico e reduzindo o tempo ~100-300x.
    """
    signal = signal.astype(np.float64)
    N = len(signal)
    p = NLM_PATCH_HALF
    M = NLM_SEARCH_HALF

    sigma_hat = _nlm_sigma(signal)
    lam = NLM_K_LAMBDA * sigma_hat
    h   = max(2.0 * NLM_PATCH_LEN * lam * lam, 1e-10)

    output = signal.copy()    # delta=0: self-similarity -> w=1
    norm   = np.ones(N)

    for delta in range(1, M + 1):
        valid_n = N - delta
        if valid_n <= 0:
            break

        # ── D_delta e prefix sum (identico ao firmware) ──────────────────
        diff_sq = (signal[delta:delta + valid_n] - signal[:valid_n]) ** 2
        cum = np.empty(N)
        cum[:valid_n] = np.cumsum(diff_sq)
        cum[valid_n:] = cum[valid_n - 1]

        # ── Indices vetorizados de todos os pares validos ─────────────────
        # i em [0, valid_n-1],  j = i + delta
        i_arr = np.arange(valid_n)          # shape (valid_n,)
        j_arr = i_arr + delta               # j < N por construcao

        # hi = min(i + p, valid_n - 1)
        hi_arr = np.minimum(i_arr + p, valid_n - 1)

        # lo = i - p - 1  (negativo -> 0 pelo operador ternario do firmware)
        lo_arr = i_arr - p - 1              # pode ser negativo

        # d2 = cum[hi] - (lo >= 0 ? cum[lo] : 0)
        cum_lo          = np.where(lo_arr >= 0, cum[lo_arr], 0.0)
        d2_arr          = cum[hi_arr] - cum_lo

        # w = exp(-d2 / h)  — vetorizado
        w_arr = np.exp(-d2_arr / h)

        # ── Acumulacao simetrica ──────────────────────────────────────────
        # output[i] += w * signal[j]
        np.add.at(output, i_arr, w_arr * signal[j_arr])
        np.add.at(norm,   i_arr, w_arr)
        # output[j] += w * signal[i]
        np.add.at(output, j_arr, w_arr * signal[i_arr])
        np.add.at(norm,   j_arr, w_arr)

    mask = norm > 1e-12
    return np.where(mask, output / norm, signal).astype(np.float32)


# ── 4. Pipeline completo ──────────────────────────────────────────────────────

def ecg_pipeline(signal):
    """Apenas Butterworth (LOESS e NLM REMOVIDOS) - ablation sem LOESS+NLM.

    Entrada: array 1-D float32, comprimento = N_POINTS
    Saida  : array 1-D float32, mesmo comprimento, filtrado
    """
    s = butterworth_apply(signal)
    # LOESS removido nesta ablation
    # NLM removido nesta ablation
    return s


# =============================================================================
# FUNCOES AUXILIARES (identicas a versao original)
# =============================================================================

def extract_acf(signal):
    """Autocorrelacao discreta normalizada - replica autocorr_compute().

    s = signal - mean(signal)
    R[k] = dot(s[:N-k], s[k:]) / dot(s, s)   k = 0..N-1
    """
    s = signal.flatten().astype(np.float64)
    n = len(s)
    s = s - s.mean()
    r0 = np.dot(s, s)
    if r0 < 1e-10:
        return np.zeros(n, dtype=np.float32)
    acf = np.empty(n, dtype=np.float32)
    for k in range(n):
        acf[k] = np.dot(s[:n - k], s[k:]) / r0
    return acf

def extract_acf_batch(X, desc=''):
    feats = []
    for i, sig in enumerate(X):
        if i % 200 == 0:
            print(f'  {desc} [{i}/{len(X)}]...', end='\r')
        feats.append(extract_acf(sig))
    print(f'  {desc} [{len(X)}/{len(X)}] OK          ')
    return np.array(feats, dtype=np.float32)

def load_domain(path_afib, path_sr, apply_filter=True):
    """Carrega CSVs de AFIB e SR, aplica pipeline de filtros se apply_filter=True.

    O pipeline (Butterworth -> LOESS -> NLM) e aplicado sinal a sinal antes
    de qualquer extracao de feature, replicando a sequencia do firmware.
    """
    X, y, fnames = [], [], []
    for path, label in [(path_afib, 1), (path_sr, 0)]:
        files = sorted(glob.glob(os.path.join(path, '*.csv')))
        if not files:
            raise RuntimeError(f'Nenhum CSV encontrado em: {path}')
        for fi, f in enumerate(files):
            if fi % 100 == 0:
                print(f'  Loading {os.path.basename(path)} [{fi}/{len(files)}]...', end='\r')
            try:
                s = pd.read_csv(f, header=None).values.flatten().astype(np.float32)
                s = s[:N_POINTS] if len(s) >= N_POINTS else np.pad(s, (0, N_POINTS - len(s)))
                if apply_filter:
                    s = ecg_pipeline(s)
                X.append(s)
                y.append(label)
                fnames.append(f)
            except Exception:
                continue
        print(f'  Loading {os.path.basename(path)} [{len(files)}/{len(files)}] OK   ')
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64), np.array(fnames)

print('Pipeline de filtros Python definido (ABLATION SEM LOESS+NLM).')
print(f'  Butterworth : HP=0.5 Hz, LP=40.0 Hz, fs=500, zero-phase (BLT)')
print(f'  LOESS       : ** REMOVIDO **')
print(f'  NLM Darbon  : ** REMOVIDO **')
print('Funcoes auxiliares (extract_acf, load_domain) definidas.')

# =============================================================================
# PATIENT-AWARE SPLIT — funções para construir vetor de patient_id por registro
# =============================================================================

_PTB_FN_RE = re.compile(r'^(\d+)_hr\.csv$')

def build_patient_groups(fnames, domain_flag, ptbxl_db_path=PTBXL_DB_CSV):
    """Constrói o vetor `groups` de identificadores de paciente para o pool combinado.
    
    - `domain_flag`: array com 0 (CH) ou 1 (PTB) por registro.
    - CSN (0): grupo sintético único por registro → 'CSN_<basename>'.
      (Zheng et al. 2020: cada arquivo do CSN corresponde a um paciente único.)
    - PTB-XL (1): grupo real 'PTB_<patient_id>' obtido de `ptbxl_database.csv`.
      O prefixo garante que IDs CSN e PTB nunca colidam.
    
    Retorna: np.array de strings (grupos), mesmo comprimento de `fnames`.
    """
    ptbxl = pd.read_csv(ptbxl_db_path, usecols=['ecg_id', 'patient_id'])
    ptbxl['ecg_id']     = ptbxl['ecg_id'].astype(int)
    ptbxl['patient_id'] = ptbxl['patient_id'].astype('Int64')
    ecg2pat = dict(zip(ptbxl['ecg_id'], ptbxl['patient_id']))

    groups = np.empty(len(fnames), dtype=object)
    unresolved = 0
    for i, (fpath, d) in enumerate(zip(fnames, domain_flag)):
        base = os.path.basename(fpath)
        if d == 0:
            groups[i] = f'CSN_{base}'
        else:
            m = _PTB_FN_RE.match(base)
            if m is None:
                groups[i] = f'PTB_ORPHAN_{base}'; unresolved += 1; continue
            pid = ecg2pat.get(int(m.group(1)))
            if pid is None or pd.isna(pid):
                groups[i] = f'PTB_ORPHAN_{base}'; unresolved += 1; continue
            groups[i] = f'PTB_{int(pid)}'
    if unresolved > 0:
        print(f'  ⚠️  {unresolved} arquivo(s) PTB sem patient_id resolvido — tratados como grupos únicos.')
    n_unique = len(set(groups))
    print(f'  Grupos únicos: {n_unique}  (para {len(groups)} registros)')
    return groups

def select_holdout_fold(splitter, X, y, groups):
    """Aplica um `Stratified*KFold` e retorna a fold cuja proporção AFIB mais
    se aproxima da proporção global — garante estratificação mesmo com grupos.
    Retorna (trainval_idx, holdout_idx).
    """
    frac_afib_global = float(y.sum()) / len(y)
    best = None; best_delta = 1.0
    for k, (tr_idx, ho_idx) in enumerate(splitter.split(X, y, groups=groups)):
        frac_ho = y[ho_idx].sum() / len(ho_idx)
        delta = abs(frac_ho - frac_afib_global)
        if delta < best_delta:
            best_delta = delta; best = (k, tr_idx, ho_idx, frac_ho)
    k, tr_idx, ho_idx, frac_ho = best
    print(f'  Fold escolhida: {k}  (fração AFIB holdout={frac_ho:.4f}, global={frac_afib_global:.4f})')
    return tr_idx, ho_idx

print('Funções patient-aware definidas: build_patient_groups(), select_holdout_fold().')


## 3. Carregamento e split patient-aware em três vias

O pool combinado CH + PTB-XL é dividido **por paciente** (`StratifiedGroupKFold`,
`groups = patient_id`), estratificado pela classe:

- **Holdout** (1 fold de 7, ≈14,3% do pool) → avaliado uma única vez, pós-treino
- **Calibração INT8** (1 fold de 17 do restante, ≈5,0% do pool) → representative dataset
- **Treino** (≈80,7% do pool) → 10-fold CV interno, também agrupado por paciente

Nenhum paciente aparece em mais de uma partição (verificado por `assert`).
Uma planilha `.csv` com todos os arquivos do holdout é salva ao final.

In [ ]:
# =============================================================================
# SPLIT PATIENT-AWARE (Revisor 4) — Ablation LOESS/NLM
# =============================================================================
# Sinais carregados com ecg_pipeline() aplicando apenas Butterworth.
# Ablation: LOESS e NLM removidos — apenas Butterworth (0.5–40 Hz).
#
# Substitui `train_test_split` por `StratifiedGroupKFold` para garantir que
# nenhum paciente apareça em mais de uma partição (treino, calibração, holdout).
#
# Reprodutibilidade: SEED=42 preservado; splits determinísticos.

# ── Carrega os dois domínios ──────────────────────────────────────────────
print('Carregando CH...')
X_ch, y_ch, fn_ch = load_domain(PATHS_CH['AFIB'], PATHS_CH['SR'])
print(f'  CH total  : {len(X_ch):5d}  AFIB={y_ch.sum():4d}  SR={(y_ch==0).sum():4d}')

print('\nCarregando PTB-XL...')
X_ptb, y_ptb, fn_ptb = load_domain(PATHS_PTB['AFIB'], PATHS_PTB['SR'])
print(f'  PTB total  : {len(X_ptb):5d}  AFIB={y_ptb.sum():4d}  SR={(y_ptb==0).sum():4d}')

# ── Concatena no pool combinado ───────────────────────────────────────────
X_all  = np.concatenate([X_ch,  X_ptb])
y_all  = np.concatenate([y_ch,  y_ptb])
fn_all = np.concatenate([fn_ch, fn_ptb])
dom_all = np.concatenate([
    np.zeros(len(y_ch),  dtype=np.int64),   # 0 = CH
    np.ones( len(y_ptb), dtype=np.int64),   # 1 = PTB
])
print(f'\nPool total combinado: {len(y_all):5d}  '
      f'AFIB={y_all.sum():4d}  SR={(y_all==0).sum():4d}')

# ── Constrói vetor de patient_id por registro ────────────────────────────
print('\nConstruindo grupos patient-aware...')
groups_all = build_patient_groups(fn_all, dom_all)

# ── 1ª divisão: separa o holdout (~14,3%) preservando paciente ───────────
print('\n1ª divisão: StratifiedGroupKFold (holdout patient-aware)...')
sgkf_ho = StratifiedGroupKFold(
    n_splits=N_SPLITS_HOLDOUT, shuffle=True, random_state=SPLIT_SEED)
idx_trainval, idx_ho = select_holdout_fold(
    sgkf_ho, X_all, y_all, groups_all)

# ── 2ª divisão: separa calibração (≈5,0% do pool) preservando paciente ───────────
print('\n2ª divisão: StratifiedGroupKFold (calibração patient-aware)...')
sgkf_cal = StratifiedGroupKFold(
    n_splits=N_SPLITS_CALIB, shuffle=True, random_state=SPLIT_SEED)
idx_train_rel, idx_calib_rel = select_holdout_fold(
    sgkf_cal,
    X_all[idx_trainval], y_all[idx_trainval], groups_all[idx_trainval])
idx_train = idx_trainval[idx_train_rel]
idx_calib = idx_trainval[idx_calib_rel]

# ── Monta os três conjuntos ───────────────────────────────────────────────
X_train_raw = X_all[idx_train]
y_train     = y_all[idx_train]
dom_train   = dom_all[idx_train]
fn_train    = fn_all[idx_train]
groups_train = groups_all[idx_train]         # <— usado no 10-fold CV interno

X_calib_raw = X_all[idx_calib]
y_calib     = y_all[idx_calib]
dom_calib   = dom_all[idx_calib]

X_ho_raw    = X_all[idx_ho]
y_ho        = y_all[idx_ho]
dom_ho      = dom_all[idx_ho]
fn_ho       = fn_all[idx_ho]

# ── Salva índices + grupos para rastreabilidade e auditoria futura ───────
with open(CALIB_IDX_PATH, 'wb') as f:
    pickle.dump({
        'idx_train': idx_train, 'idx_calib': idx_calib, 'idx_ho': idx_ho,
        'groups_all': groups_all,   # NOVO: para verificação patient-level ex post
    }, f)

# ── Máscaras por domínio no holdout ──────────────────────────────────────
mask_ch  = dom_ho == 0
mask_ptb = dom_ho == 1

print(f'\n{"─"*60}')
print(f'  Split três vias patient-aware:')
print(f'  Treino  : {len(y_train):5d}  '
      f'AFIB={y_train.sum():4d}  SR={(y_train==0).sum():4d}  '
      f'CH={(dom_train==0).sum():4d}  PTB={(dom_train==1).sum():4d}')
print(f'  Calibr. : {len(y_calib):5d}  '
      f'AFIB={y_calib.sum():4d}  SR={(y_calib==0).sum():4d}  '
      f'CH={(dom_calib==0).sum():4d}  PTB={(dom_calib==1).sum():4d}')
print(f'  Holdout : {len(y_ho):5d}  '
      f'AFIB={y_ho.sum():4d}  SR={(y_ho==0).sum():4d}  '
      f'CH={mask_ch.sum():4d}  PTB={mask_ptb.sum():4d}')
print(f'  Ratio AFIB:SR no holdout: 1:{(y_ho==0).sum()/max(y_ho.sum(),1):.1f}')
print(f'{"─"*60}')

# ── Verificações patient-level (obrigatórias) ────────────────────────────
g_train = set(groups_all[idx_train])
g_calib = set(groups_all[idx_calib])
g_ho    = set(groups_all[idx_ho])
assert g_train.isdisjoint(g_ho),    'ERRO: paciente em treino e holdout.'
assert g_train.isdisjoint(g_calib), 'ERRO: paciente em treino e calibração.'
assert g_calib.isdisjoint(g_ho),    'ERRO: paciente em calibração e holdout.'
print('  ✅ Verificação patient-level: as três partições são disjuntas por paciente.')

# ── Verificação flexível de proporções (informativa, não bloqueante) ─────
frac_ho = len(y_ho) / len(y_all)
frac_afib_ho = y_ho.sum() / len(y_ho) if len(y_ho) > 0 else 0
print(f'  Fração holdout      : {frac_ho:.4f} (alvo ~0.15)')
print(f'  Fração AFIB holdout : {frac_afib_ho:.4f} (esperado ~0.117)')
assert 0.12 <= frac_ho <= 0.18, f'Fração holdout fora do intervalo aceitável: {frac_ho:.4f}'

# ── Planilha do holdout patient-aware ───────────────────────────────────────────
holdout_df = pd.DataFrame({
    'index_pool':  list(idx_ho),
    'filename':    [os.path.basename(f) for f in fn_ho],
    'filepath':    list(fn_ho),
    'label':       [{0:'SR', 1:'AFIB'}[int(l)] for l in y_ho],
    'dataset':     ['CH' if d==0 else 'PTB' for d in dom_ho],
    'label_int':   list(y_ho),
    'patient_group': list(groups_all[idx_ho]),
})
holdout_df.to_csv(HOLDOUT_CSV_PATH, index=False)
print(f'\nPlanilha holdout patient-aware salva: {HOLDOUT_CSV_PATH}')
print(f'  Linhas: {len(holdout_df)}   Colunas: {list(holdout_df.columns)}')
print(holdout_df.groupby(['dataset', 'label']).size().rename('n')
      .reset_index().to_string(index=False))

# ── OAT: garante que o teste é exatamente o da referência (comparação pareada) ──
_ref_idx = os.path.join(REF_DIR, 'ablation_loess_nlm_calib_indices.pkl')
with open(_ref_idx, 'rb') as f: _ref = pickle.load(f)
for _k in ('idx_train', 'idx_calib', 'idx_ho'):
    assert np.array_equal(np.asarray(_ref[_k]), np.asarray(locals()[_k])), f'Partição {_k} difere da referência'
print('  ✅ Partições idênticas às da referência (teste pareado garantido).')


In [ ]:

import json, hashlib
from datetime import datetime
from sklearn.metrics import confusion_matrix, roc_auc_score

KEYS = ['AUC', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1-Score', 'Accuracy']
def _compute(p, y, thr):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {'AUC': roc_auc_score(y, p), 'Sensitivity': tp/(tp+fn), 'Specificity': tn/(tn+fp),
            'PPV': tp/(tp+fp) if (tp+fp) else np.nan, 'NPV': tn/(tn+fn) if (tn+fn) else np.nan,
            'F1-Score': 2*tp/(2*tp+fp+fn), 'Accuracy': (tp+tn)/len(y),
            'TP': int(tp), 'FN': int(fn), 'FP': int(fp), 'TN': int(tn)}
def _boot(p, y, thr, n=1000, seed=42):
    rng = np.random.RandomState(seed); b = {k: [] for k in KEYS}
    for _ in range(n):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        m = _compute(p[i], y[i], thr)
        for k in KEYS: b[k].append(m[k])
    return {k: [float(np.nanpercentile(b[k], 2.5)), float(np.nanpercentile(b[k], 97.5))] for k in KEYS}

def save_float_outputs(out_dir, label, p_ho, tau, n_params, model_bytes, model_sha, extra=None):
    os.makedirs(out_dir, exist_ok=True)
    p_ho = np.asarray(p_ho, dtype=float).ravel(); y = np.asarray(y_ho, dtype=int).ravel()
    assert len(p_ho) == len(y) == 4025
    pt, ci = _compute(p_ho, y, tau), _boot(p_ho, y, tau)
    rec = {'variant': label, 'precision': 'float32 (no INT8 conversion)', 'timestamp': datetime.now().isoformat(),
           'holdout': {'n_total': int(len(y)), 'n_afib': int(y.sum()), 'n_sr': int((y == 0).sum())},
           'thresholds': {'tau_float32_youden': float(tau), 'tau_int8_firmware': float(tau)},
           'int8_with_tau_firmware': {k: float(pt[k]) for k in KEYS}, 'int8_ci95': ci,
           'int8_confusion': {k: pt[k] for k in ['TP', 'FN', 'FP', 'TN']},
           'n_params': int(n_params), 'tflite_bytes': int(model_bytes),
           'sha256': {'model': model_sha, 'tflite': None}, 'extra': extra or {}}
    json.dump(rec, open(os.path.join(out_dir, 'metrics_final.json'), 'w'), indent=2)
    pd.DataFrame({'filename': [os.path.basename(str(x)) for x in fn_ho],
                  'dataset': np.where(np.asarray(dom_ho) == 0, 'CH', 'PTB'),
                  'label_int': y, 'p_int8': p_ho, 'pred_int8': (p_ho >= tau).astype(int)}
                 ).to_csv(os.path.join(out_dir, 'holdout_probs_int8.csv'), index=False)
    print(f"{label}: AUC={pt['AUC']:.4f} [{ci['AUC'][0]:.4f}, {ci['AUC'][1]:.4f}]  "
          f"Sens={pt['Sensitivity']:.4f}  Spec={pt['Specificity']:.4f}  F1={pt['F1-Score']:.4f}  tau={tau:.4f}")

import pickle, time
from scipy.signal import find_peaks
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_curve

def detect_r(x, fs=FS):
    """Detecção de picos R estilo Pan–Tompkins sobre o sinal Butterworth."""
    x = np.asarray(x, dtype=float).ravel()
    e = np.diff(x, prepend=x[0]) ** 2
    w = int(0.15 * fs)
    mwi = np.convolve(e, np.ones(w) / w, mode='same')
    pk, _ = find_peaks(mwi, height=0.3 * np.percentile(mwi, 99), distance=int(0.25 * fs))
    h = int(0.075 * fs); r = []
    for p in pk:
        a, b = max(0, p - h), min(len(x), p + h)
        r.append(a + int(np.argmax(np.abs(x[a:b]))))
    return np.unique(np.array(r, dtype=int))

FEAT_NAMES = ['n_beats', 'mean_rr', 'sd_rr', 'rmssd', 'pnn50', 'cv_rr', 'min_rr', 'max_rr',
              'mad_rr', 'mean_abs_drr_norm', 'shannon_entropy', 'turning_point_ratio']
def rr_features(x, fs=FS):
    r = detect_r(x, fs); rr = np.diff(r) / fs
    rr = rr[(rr > 0.25) & (rr < 2.5)]
    if len(rr) < 3: return np.full(len(FEAT_NAMES), np.nan)
    d = np.diff(rr)
    hist, _ = np.histogram(rr, bins=16, range=(0.25, 2.5)); pr = hist[hist > 0] / hist.sum()
    tp = np.sum((rr[1:-1] > rr[:-2]) & (rr[1:-1] > rr[2:]) | (rr[1:-1] < rr[:-2]) & (rr[1:-1] < rr[2:])) / max(len(rr) - 2, 1)
    return np.array([len(r), rr.mean(), rr.std(), np.sqrt(np.mean(d ** 2)), np.mean(np.abs(d) > 0.05),
                     rr.std() / rr.mean(), rr.min(), rr.max(), np.median(np.abs(rr - np.median(rr))),
                     np.mean(np.abs(d)) / rr.mean(), -np.sum(pr * np.log2(pr)), tp])

t0 = time.time()
F_train = np.stack([rr_features(x) for x in X_train_raw])
F_ho    = np.stack([rr_features(x) for x in X_ho_raw])
n_bad_tr, n_bad_ho = int(np.isnan(F_train[:, 0]).sum()), int(np.isnan(F_ho[:, 0]).sum())
med = np.nanmedian(F_train, axis=0)
for F in (F_train, F_ho):
    bad = np.isnan(F); F[bad] = np.take(med, np.where(bad)[1])
print(f'Features R-R: treino {F_train.shape}, teste {F_ho.shape} ({time.time()-t0:.0f} s)')
print(f'  Registros com < 3 intervalos R-R válidos (features imputadas pela mediana do treino): treino {n_bad_tr}, teste {n_bad_ho}')

MODELS = {
    'base_rf_rr':  ('Random forest, R-R features',
                    lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced',
                                                   random_state=SPLIT_SEED, n_jobs=-1)),
    'base_svm_rr': ('RBF-SVM, R-R features',
                    lambda: make_pipeline(StandardScaler(), SVC(C=1.0, gamma='scale', class_weight='balanced',
                                                                probability=True, random_state=SPLIT_SEED))),
}
kf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
for name, (label, make) in MODELS.items():
    t0 = time.time()
    oof = np.zeros(len(y_train))
    for tr, va in kf.split(F_train, y_train, groups=groups_train):
        m = make().fit(F_train[tr], y_train[tr]); oof[va] = m.predict_proba(F_train[va])[:, 1]
    fpr, tpr, thr = roc_curve(y_train, oof); tau = float(thr[np.argmax(tpr - fpr)])
    model = make().fit(F_train, y_train)
    p_ho = model.predict_proba(F_ho)[:, 1]
    blob = pickle.dumps(model)
    if name == 'base_rf_rr':
        n_par = int(sum(est.tree_.node_count for est in model.estimators_))
    else:
        svc = model[-1]; n_par = int(svc.support_vectors_.size + svc.dual_coef_.size)
    save_float_outputs(os.path.join(OAT_ROOT, name), label, p_ho, tau, n_par, len(blob),
                       hashlib.sha256(blob).hexdigest(), {'features': FEAT_NAMES, 'train_s': round(time.time()-t0, 1)})
